# 🧠 การหาค่าเหมาะที่สุดด้วย Weight Decay และ Regularization

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Weight Decay**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายคณิตศาสตร์ของ L2 Regularization, shrinkage factor $(1 - \alpha \lambda)$ และ Decoupled Weight Decay (AdamW)
2. สร้างชุดข้อมูลขนาดเล็กที่มีสัญญาณรบกวน (noise) และปรับโมเดลการถดถอยพหุนามดีกรีสูง (high-degree polynomial regression)
3. เปรียบเทียบโมเดลที่ปรับโดยมีและไม่มี L2 regularization (โดยใช้ Ridge regression) เพื่อดูว่า weight decay ป้องกันการเรียนรู้เกิน (overfitting) อย่างไร
4. แสดงภาพ (visualize) ว่า weight decay ย่อขนาดค่าสัมประสิทธิ์ของพารามิเตอร์ (ขนาดน้ำหนัก) ให้เข้าใกล้ศูนย์ได้อย่างไร
5. เขียนโค้ด Stochastic Gradient Descent (SGD) พร้อมกับ Weight Decay ขึ้นมาจากศูนย์ด้วย Python
6. เชื่อมโยงแนวคิดเหล่านี้เข้ากับพารามิเตอร์การฝึกสอนเริ่มต้นของ YOLO `weight_decay=0.0005`

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อน

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge

# Set seed for reproducibility
np.random.seed(42)

## 1. การสร้างข้อมูลการฝึกสอนที่มีสัญญาณรบกวน (Noisy Training Data)

เราสร้างชุดข้อมูลขนาดเล็กที่มีตัวอย่างการฝึกสอนที่มีสัญญาณรบกวนจำนวน 12 ตัวอย่าง ซึ่งสุ่มมาจาก $y = \cos(1.5 \pi x)$ เนื่องจากขนาดตัวอย่างมีขนาดเล็ก พหุนามดีกรีสูงจะเกิดการเรียนรู้เกิน (overfit) ได้ง่าย

In [ ]:
def true_func(x):
    return np.cos(1.5 * np.pi * x)

n_samples = 12
x_train = np.sort(np.random.rand(n_samples))
y_train = true_func(x_train) + np.random.normal(0, 0.15, n_samples)

x_test = np.linspace(0, 1, 100)
y_true = true_func(x_test)

## 2. การปรับพหุนาม: แบบที่ไม่มี Regularization vs. แบบที่มี Regularization (Weight Decay)

มาปรับการถดถอยพหุนามดีกรี 10 (Degree 10 polynomial regression) โดยใช้:
1.  **Standard Linear Regression (ไม่มี Weight Decay):** จับสัญญาณรบกวน ทำให้เกิดความแปรปรวนสูง (high variance)
2.  **Ridge Regression (มี Weight Decay / L2 Regularization, $\lambda = 0.05$):** เพิ่มบทลงโทษ (penalty) เพื่อย่อขนาดน้ำหนักลง

In [ ]:
degree = 10

# Model 1: No Regularization
model_no_reg = make_pipeline(PolynomialFeatures(degree), LinearRegression())
model_no_reg.fit(x_train[:, np.newaxis], y_train)
y_pred_no_reg = model_no_reg.predict(x_test[:, np.newaxis])

# Model 2: Ridge Regularization
model_reg = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=0.05))
model_reg.fit(x_train[:, np.newaxis], y_train)
y_pred_reg = model_reg.predict(x_test[:, np.newaxis])

# Plotting the comparisons
plt.figure(figsize=(12, 6))
plt.plot(x_test, y_true, color='black', linewidth=2.5, label='True Function f(x)')
plt.plot(x_test, y_pred_no_reg, color='red', linestyle='--', linewidth=2, label='No Weight Decay (Overfitted)')
plt.plot(x_test, y_pred_reg, color='teal', linewidth=3, label='With Weight Decay (Regularized)')
plt.scatter(x_train, y_train, color='orange', edgecolor='k', s=55, zorder=5, label='Training Points')
plt.ylim(-2.5, 2.5)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Weight Decay (L2) Regularization on Degree 10 Polynomial')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

จากการสังเกต:
-   **เส้นสีแดง (ไม่มี Weight Decay):** โค้งงออย่างรุนแรงที่ขอบเพื่อที่จะผ่านจุดข้อมูลฝึกสอนที่มีสัญญาณรบกวนพอดี ส่งผลให้การสรุปความรู้ทั่วไปแย่ลง (poor generalisation)
-   **เส้นสีน้ำเงินแกมเขียว (มี Weight Decay):** จำกัดน้ำหนักไม่ให้ขยายใหญ่เกินไป ทำให้เส้นโค้งเรียบและสอดคล้องกับฟังก์ชันสร้างข้อมูลที่แท้จริงอย่างมาก

## 3. การแสดงภาพการหดตัวของสัมประสิทธิ์น้ำหนัก (Weight Coefficient Shrinkage)

มาตรวจสอบน้ำหนัก (สัมประสิทธิ์) ของทั้งสองโมเดลกัน

In [ ]:
coef_no_reg = np.abs(model_no_reg.named_steps['linearregression'].coef_)
coef_reg = np.abs(model_reg.named_steps['ridge'].coef_)

coef_no_reg = coef_no_reg[1:]
coef_reg = coef_reg[1:]

indices = np.arange(1, len(coef_no_reg) + 1)

plt.figure(figsize=(12, 5))
plt.bar(indices - 0.2, coef_no_reg, width=0.4, color='red', label='No Weight Decay')
plt.bar(indices + 0.2, coef_reg, width=0.4, color='teal', label='With Weight Decay')
plt.yscale('log')
plt.xlabel('Polynomial Term Index')
plt.ylabel('Absolute Coefficient Magnitude (Log Scale)')
plt.title('Effect of Weight Decay on Parameter Coefficients')
plt.xticks(indices)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3, which='both')
plt.show()

สังเกตว่าสัมประสิทธิ์ของโมเดลที่ไม่มี regularization นั้นมีขนาดใหญ่มาก ในขณะที่โมเดลที่มี regularization จะช่วยบีบสัมประสิทธิ์ให้เล็กลงได้

## 4. การสร้าง Weight Decay ขึ้นมาจากศูนย์

มาเขียนฟังก์ชัน Python เพื่ออัปเดตเมทริกซ์น้ำหนักโดยใช้ SGD ร่วมกับ Weight Decay:
$$\mathbf{w}_{t+1} = \mathbf{w}_t(1 - \alpha \lambda) - \alpha \nabla E_0(\mathbf{w}_t)$$

In [ ]:
def update_weights_with_decay(w, grad, lr, weight_decay):
    shrinkage = 1.0 - lr * weight_decay
    w_updated = w * shrinkage - lr * grad
    return w_updated

w_test = np.array([2.5, -1.5])
grad_test = np.array([0.4, -0.2])
print("Updated weights:", update_weights_with_decay(w_test, grad_test, lr=0.1, weight_decay=0.01))

## 💡 การเชื่อมโยงไปยัง YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **Decoupled Weight Decay (AdamW):** เมื่อฝึกสอนโมเดล YOLO คุณต้องกำหนดค่า `weight_decay=0.0005` ในการตั้งค่า เนื่องจาก YOLO ใช้ **AdamW** เป็นหลักค่าเริ่มต้น ดังนั้น weight decay จึงถูกนำไปใช้โดยตรงกับพารามิเตอร์ แทนที่จะนำไปใช้ผ่านค่าโมเมนตัมของการเคลื่อนที่ของเกรเดียนต์
*   การหดตัวแบบแยกส่วน (decoupled shrinkage) นี้ช่วยป้องกันไม่ให้น้ำหนักมีขนาดใหญ่เกินไป หลีกเลี่ยงปัญหาการระเบิดทางตัวเลข (น้ำหนักกลายเป็น `NaN`) และทำให้แน่ใจว่าแบบจำลองจะสรุปความรู้ทั่วไปไปยังชุดข้อมูลการตรวจสอบความถูกต้องได้ดี